# EDSS 남은 식별자 공백 처리 검산

## TL;DR

취업통계 2023–2024년 집계 구간과 고위험 미연결 패널 4개를 재검산한다. 고위험 패널의 수동 검토 잔여는 0개이며, 취업 후보 ID는 검토용 증거일 뿐 정식 `개방ID`로 대입하지 않는다.

## Context & Methods

- 취업통계 2010–2022년의 OpenID별 학과·단과대 서명을 2023–2024년 학교·연도별 서명과 완전 일치 비교한다.
- 서명 크기가 3 미만이거나 복수 OpenID와 일치하면 미해결로 둔다.
- 단일 후보도 같은 연도의 0101 시도·본분교 맥락을 추가 확인한다.
- 파생 파일에는 2023–2024년 공개 집계 필드와 출처 추적 필드만 둔다.
- 고위험 패널의 미연결 키는 기준기간 경계와 내부 공백으로 분리한다.
- 승인된 내부 공백 판정은 기대 행 수·공식 근거·비대입 처리 규칙을 검증한 뒤 별도 판정층으로 적용한다.

## Data

In [1]:
from pathlib import Path
from collections import Counter
import csv, gzip, hashlib, json
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'data/metadata').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/metadata').exists(), 'repository root not found'
summary_path = ROOT / 'data/metadata/edss_remaining_identity_gap_resolution.json'
candidate_path = ROOT / 'data/metadata/edss_employment_2023_2024_open_id_candidates.csv'
review_path = ROOT / 'data/metadata/edss_high_orphan_panel_review.csv'
decision_path = ROOT / 'data/metadata/edss_high_orphan_manual_decisions.csv'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
derived_path = ROOT / summary['outputs']['derived_employment']['path']
with candidate_path.open(encoding='utf-8-sig', newline='') as handle:
    candidates = list(csv.DictReader(handle))
with review_path.open(encoding='utf-8-sig', newline='') as handle:
    reviews = list(csv.DictReader(handle))
with decision_path.open(encoding='utf-8-sig', newline='') as handle:
    manual_decisions = list(csv.DictReader(handle))
with gzip.open(derived_path, 'rt', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    derived_fields = reader.fieldnames
    derived_rows = sum(1 for _ in reader)
derived_sha256 = hashlib.sha256(derived_path.read_bytes()).hexdigest()
display(Markdown(f'입력 요약 1개, 후보 상태 {len(candidates):,}개, 고위험 패널 {len(reviews)}개, 수동 판정 {len(manual_decisions)}개를 읽었다.'))

입력 요약 1개, 후보 상태 3,281개, 고위험 패널 4개, 수동 판정 1개를 읽었다.

## Results

In [2]:
employment = summary['employment']
high = summary['high_orphan_panels']
status_counts = Counter(row['resolution_status'] for row in candidates)
forbidden_individual_fields = {
    '성명', '주민등록번호', '외국인등록번호', '생년월일', '성별', '전화번호', '이메일', '주소'
}
assert derived_rows == 46_962
assert derived_rows == summary['outputs']['derived_employment']['row_count']
assert derived_sha256 == summary['outputs']['derived_employment']['sha256']
assert employment['canonical_open_id_imputed_row_count'] == 0
assert '개방ID' not in derived_fields
assert forbidden_individual_fields.isdisjoint(derived_fields)
assert sum(status_counts.values()) == 3_281
assert status_counts['candidate_signature_context_confirmed'] == 30
assert status_counts['candidate_0101_context_conflict'] == 2
rows = [
    ('파생 취업 집계 행', f'{derived_rows:,}'),
    ('학교·연도 상태', f'{len(candidates):,}'),
    ('0101 맥락 일치 후보', f"{status_counts['candidate_signature_context_confirmed']:,}"),
    ('0101 맥락 충돌 후보', f"{status_counts['candidate_0101_context_conflict']:,}"),
    ('정식 개방ID 대입', '0'),
    ('개인 필드 검출', '0'),
]
table = '| 검산 항목 | 결과 |\n|---|---:|\n' + '\n'.join(f'| {k} | {v} |' for k, v in rows)
display(Markdown(table))

| 검산 항목 | 결과 |
|---|---:|
| 파생 취업 집계 행 | 46,962 |
| 학교·연도 상태 | 3,281 |
| 0101 맥락 일치 후보 | 30 |
| 0101 맥락 충돌 후보 | 2 |
| 정식 개방ID 대입 | 0 |
| 개인 필드 검출 | 0 |

In [3]:
assert len(reviews) == 4
assert high['explained_temporal_boundary_panel_count'] == 3
assert high['status'] == 'complete'
assert high['manual_review_required_panel_count'] == 0
assert high['manual_review_keys'] == []
assert high['manually_resolved_panel_count'] == 1
assert high['manually_resolved_key_count'] == 1
assert len(manual_decisions) == 1
decision = manual_decisions[0]
assert decision['catalog_code'] == '1209'
assert decision['year'] == '2019'
assert decision['open_id'] == '5831784427'
assert int(decision['expected_row_count']) == 16
assert decision['decision_status'] == 'approved'
assert decision['recommended_handling'].startswith('retain_same_open_id')
table = '| 코드 | 데이터셋 | 판정 | 경계 키 | 내부 공백 키 | 수동 확정 키·행 |\n|---|---|---|---:|---:|---:|\n'
table += '\n'.join(
    f"| {r['catalog_code']} | {r['dataset']} | {r['review_disposition']} | {r['boundary_key_count']} | {r['internal_gap_key_count']} | {r['manually_resolved_key_count']}·{r['manually_resolved_row_count']} |"
    for r in reviews
)
display(Markdown(table))

| 코드 | 데이터셋 | 판정 | 경계 키 | 내부 공백 키 | 수동 확정 키·행 |
|---|---|---|---:|---:|---:|
| 0202 | 대학입학전형기본계획_전문대학 | explained_temporal_boundary | 26 | 0 | 0·0 |
| 0204 | 대학입학전형시행계획_전문대학 | explained_temporal_boundary | 12 | 0 | 0·0 |
| 1102 | 도서관예산현황 | explained_temporal_boundary | 35 | 0 | 0·0 |
| 1209 | 법인임원현황 | explained_manual_identity_scope_decision | 66 | 1 | 1·16 |

## Takeaways

- 2023–2024년 취업통계 46,962행은 공개 집계 스키마로 안전하게 분리됐다.
- 30개 학교·연도 후보는 학과서명과 0101 맥락이 모두 맞지만 공식 교차표가 아니므로 확정 ID가 아니다.
- 0202·0204·1102의 고위험 미연결은 모두 기준기간 경계로 설명된다.
- 1209의 2019년 내부 공백 1개·16행은 남인천캠퍼스의 학위과정 범위 차이로 확정했다. 동일 OpenID와 원본 행을 보존하며 대입·보간하지 않는다.
- 고위험 패널 4개의 수동 검토 잔여는 0개다. 전체 요약의 `review_required`는 취업통계 공식 교차표 미확보 때문에 유지한다.